<a href="https://colab.research.google.com/github/Tamannapanwar17/Artificial-Neural-Network/blob/main/minor_Basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle (1).json


{'kaggle (1).json': b'{"username":"tamannapanwar","key":"513b7290bd35147190485641dc827d96"}'}

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
!kaggle datasets download -d paultimothymooney/breast-histopathology-images

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/breast-histopathology-images
License(s): CC0-1.0
100% 3.10G/3.10G [02:49<00:00, 19.7MB/s]



In [5]:
!unzip -q breast-histopathology-images.zip -d /content/breast_histopathology

In [6]:
import os
import glob
import shutil
import random

random.seed(42)

SOURCE = "/content/breast_histopathology/IDC_regular_ps50_idx5"
DEST = "/content/ckks_dataset"

# Create folders
for split in ["train", "validation", "test"]:
    for label in ["0", "1"]:
        os.makedirs(
            os.path.join(DEST, split, label),
            exist_ok=True
        )

# Collect images from class 0 and class 1
class_0 = glob.glob(os.path.join(SOURCE, "**", "0", "*.png"), recursive=True)
class_1 = glob.glob(os.path.join(SOURCE, "**", "1", "*.png"), recursive=True)

print("Class 0 images:", len(class_0))
print("Class 1 images:", len(class_1))

Class 0 images: 198738
Class 1 images: 78786


In [7]:
random.shuffle(class_0)
random.shuffle(class_1)

class_0 = class_0[:100]
class_1 = class_1[:100]

print("Selected class 0:", len(class_0))
print("Selected class 1:", len(class_1))

Selected class 0: 100
Selected class 1: 100


In [8]:
def split_images(images):
    train = images[:80]
    validation = images[80:90]
    test = images[90:100]

    return train, validation, test

train_0, val_0, test_0 = split_images(class_0)
train_1, val_1, test_1 = split_images(class_1)

In [9]:
def copy_images(images, destination):
    for i, image_path in enumerate(images):
        filename = f"{i:04d}.png"
        shutil.copy(
            image_path,
            os.path.join(destination, filename)
        )

# Class 0
copy_images(train_0, f"{DEST}/train/0")
copy_images(val_0, f"{DEST}/validation/0")
copy_images(test_0, f"{DEST}/test/0")

# Class 1
copy_images(train_1, f"{DEST}/train/1")
copy_images(val_1, f"{DEST}/validation/1")
copy_images(test_1, f"{DEST}/test/1")

In [10]:
# final dataset
for split in ["train", "validation", "test"]:
    for label in ["0", "1"]:
        path = os.path.join(DEST, split, label)
        count = len(os.listdir(path))
        print(f"{split:10s} class {label}: {count}")

train      class 0: 80
train      class 1: 80
validation class 0: 10
validation class 1: 10
test       class 0: 10
test       class 1: 10


In [12]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import os

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [13]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [14]:
IMG_SIZE = (50, 50)
BATCH_SIZE = 16

train_dataset = tf.keras.utils.image_dataset_from_directory(
    "/content/ckks_dataset/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=42
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    "/content/ckks_dataset/validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    "/content/ckks_dataset/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

Found 160 files belonging to 2 classes.
Found 20 files belonging to 2 classes.
Found 20 files belonging to 2 classes.


In [15]:
print(train_dataset.class_names)

['0', '1']


In [16]:
for images, labels in train_dataset.take(1):
    print(images.shape)
    print(labels.shape)

(16, 50, 50, 3)
(16, 1)


# Build cnn model

In [17]:
# 50×50×3
#    ↓
# Conv2D (kernel = 3, padding = same,16)
#    ↓
# ReLU
#    ↓
# MaxPooling (2X2)
#    ↓
# Conv2D (32)
#    ↓
# ReLU remaining all same
#    ↓
# MaxPooling
#    ↓
# Flatten
#    ↓
# Dense (64, relu) hidden -> (1,sigmoid)
#    ↓
# Output

In [18]:
model = keras.Sequential([

    layers.Input(shape=(50, 50, 3)),

    # Normalize pixel values
    layers.Rescaling(1./255),

    # CNN layer 1
    layers.Conv2D(
        16,
        kernel_size=3,
        padding="same",
        activation="relu"
    ),

    layers.MaxPooling2D(pool_size=2),

    # CNN layer 2
    layers.Conv2D(
        32,
        kernel_size=3,
        padding="same",
        activation="relu"
    ),

    layers.MaxPooling2D(pool_size=2),

    # Convert feature maps into vector
    layers.Flatten(),

    # Fully connected layer
    layers.Dense(
        64,
        activation="relu"
    ),

    # Binary classification
    layers.Dense(
        1,
        activation="sigmoid"
    )
])

In [19]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 50, 50, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 50, 50, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 25, 25, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 25, 25, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       294,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 300,129 (1.14 MB)

 Trainable params: 300,129 (1.14 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [114]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=10
)

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 1.0000 - loss: 9.3000e-04 - val_accuracy: 0.7000 - val_loss: 0.7114
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 8.9351e-04 - val_accuracy: 0.7000 - val_loss: 0.7023
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 8.6335e-04 - val_accuracy: 0.7000 - val_loss: 0.6979
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 8.6893e-04 - val_accuracy: 0.7000 - val_loss: 0.7025
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 8.7987e-04 - val_accuracy: 0.7000 - val_loss: 0.7047
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 7.9344e-04 - val_accuracy: 0.6500 - val_loss: 0.6982
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 8.3402e-04 - val_accuracy: 0.7000 - val_loss: 0.7127
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 7.8732e-0

In [115]:
test_loss, test_accuracy = model.evaluate(test_dataset)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7500 - loss: 1.0527
Test loss: 1.0527011156082153
Test accuracy: 0.75


In [23]:
images, labels = next(iter(test_dataset))

image = images[0]
true_label = labels[0].numpy()[0]

print("Image shape:", image.shape)
print("True label:", true_label)

Image shape: (50, 50, 3)
True label: 0.0


In [24]:
image_batch = tf.expand_dims(image, axis=0)

In [25]:
prediction = model.predict(image_batch)

print("Prediction probability:", prediction[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 740ms/step
Prediction probability: 0.31433696


In [26]:
predicted_class = 1 if prediction[0][0] >= 0.5 else 0

print("Predicted class:", predicted_class)
print("True class:", int(true_label))

Predicted class: 0
True class: 0


In [27]:
model.save("/content/cnn_model.keras")

In [28]:
print(os.path.exists("/content/cnn_model.keras"))

True


In [29]:
import tensorflow as tf

model = tf.keras.models.load_model("/content/cnn_model.keras")

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 50, 50, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 50, 50, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 25, 25, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 25, 25, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       294,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 900,389 (3.43 MB)

 Trainable params: 300,129 (1.14 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 600,260 (2.29 MB)

In [30]:
!pip install tenseal

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 104.5 MB/s eta 0:00:00


In [31]:
import tenseal as ts

In [116]:
# This creates the cryptographic environment required for CKKS operations.
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)

context.global_scale = 2**40

In [35]:
# Basic operations
x = [1.5, 2.5, 3.5]

encrypted_x = ts.ckks_vector(context, x)

encrypted_result = encrypted_x * 2

decrypted_result = encrypted_result.decrypt()

print(decrypted_result)

[3.0000004010503174, 5.00000067060542, 7.000000936597486]


In [36]:
# Take one test image
images, labels = next(iter(test_dataset))

image = images[0]
label = labels[0].numpy()[0]

print("Image shape:", image.shape)
print("True label:", int(label))

Image shape: (50, 50, 3)
True label: 0


In [37]:
# CKKS works with vectors of numerical values, so flatten it:
image_vector = tf.reshape(image, [-1]).numpy()

print("Vector shape:", image_vector.shape)
print("Number of values:", len(image_vector))
print("First 10 values:", image_vector[:10])

Vector shape: (7500,)
Number of values: 7500
First 10 values: [238. 217. 227. 208. 174. 198. 226. 192. 213. 223.]


In [38]:
image_vector = image_vector / 255.0

print("First 10 normalized values:")
print(image_vector[:10])

First 10 normalized values:
[0.93333334 0.8509804  0.8901961  0.8156863  0.68235296 0.7764706
 0.8862745  0.7529412  0.8352941  0.8745098 ]


In [117]:
# This creates the cryptographic environment required for CKKS operations.
context.generate_galois_keys()

In [41]:
encrypted_image = ts.ckks_vector(
    context,
    image_vector.tolist()
)

print("Image encrypted successfully.")

Image encrypted successfully.


In [42]:
decrypted_image = encrypted_image.decrypt()

print("First 10 original values:")
print(image_vector[:10])

print("\nFirst 10 decrypted values:")
print(decrypted_image[:10])

First 10 original values:
[0.93333334 0.8509804  0.8901961  0.8156863  0.68235296 0.7764706
 0.8862745  0.7529412  0.8352941  0.8745098 ]

First 10 decrypted values:
[0.9333333409334481, 0.8509804012037566, 0.8901960841924013, 0.8156862860867149, 0.6823529554331249, 0.776470600351743, 0.8862745185677794, 0.7529411871586124, 0.835294127542918, 0.8745098078753825]


In [43]:
# Test encrypted arithmetic
encrypted_result = encrypted_image * 2

# Decrypt the result
result = encrypted_result.decrypt()

print("First 10 original values:")
print(image_vector[:10])

print("\nFirst 10 values after encrypted × 2:")
print(result[:10])

First 10 original values:
[0.93333334 0.8509804  0.8901961  0.8156863  0.68235296 0.7764706
 0.8862745  0.7529412  0.8352941  0.8745098 ]

First 10 values after encrypted × 2:
[1.8666693548409912, 1.7019632371038573, 1.7803947195301746, 1.631374912587734, 1.3647078634917214, 1.5529434225772196, 1.7725515704628734, 1.505884531598965, 1.670590646042038, 1.749022116904308]


In [44]:
# Test encrypted addition
encrypted_result = encrypted_image + 0.5

result = encrypted_result.decrypt()

print("First 10 values:")
print(result[:10])

First 10 values:
[1.4333333409334483, 1.3509804012037563, 1.390196084192401, 1.3156862860867156, 1.182352955433125, 1.2764706003517432, 1.386274518567779, 1.2529411871586125, 1.335294127542918, 1.3745098078753826]


In [45]:
# Get the Dense layer weights
for layer in model.layers:
    print(layer.name, layer.__class__.__name__)

rescaling Rescaling
conv2d Conv2D
max_pooling2d MaxPooling2D
conv2d_1 Conv2D
max_pooling2d_1 MaxPooling2D
flatten Flatten
dense Dense
dense_1 Dense


In [46]:
# Now extract the first Dense layer:
dense_layer = model.get_layer("dense")

weights, bias = dense_layer.get_weights()

print("Weights shape:", weights.shape)
print("Bias shape:", bias.shape)

Weights shape: (4608, 64)
Bias shape: (64,)


In [47]:
# Take one image
images, labels = next(iter(test_dataset))
image = images[0]

# Normalize
image = image / 255.0

# Flatten
x = tf.reshape(image, [-1]).numpy()

print("Input size:", len(x))

Input size: 7500


For CKKS, we will initially use a simpler architecture based mainly on matrix multiplication + addition + polynomial activation.

In [66]:
import tensorflow as tf
import numpy as np
import random
import os

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.environ["TF_DETERMINISTIC_OPS"] = "1"

In [67]:
from tensorflow import keras
from tensorflow.keras import layers

ckks_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(50, 50, 3)),
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32),
    tf.keras.layers.Lambda(lambda x: x * x),
    tf.keras.layers.Dense(16),
    tf.keras.layers.Lambda(lambda x: x * x),
    tf.keras.layers.Dense(1)
])

ckks_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [68]:
history = ckks_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=100
)
# 10 - 0.5
# 300 - 120.578857421875
# Test accuracy: 0.699999988079071

Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 87ms/step - accuracy: 0.4625 - loss: 1.8086 - val_accuracy: 0.5000 - val_loss: 0.6804
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4688 - loss: 0.8072 - val_accuracy: 0.5000 - val_loss: 0.8931
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4750 - loss: 0.6849 - val_accuracy: 0.5500 - val_loss: 0.6898
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5000 - loss: 0.6649 - val_accuracy: 0.5500 - val_loss: 0.6792
Epoch 5/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5125 - loss: 0.7169 - val_accuracy: 0.5000 - val_loss: 0.6782
Epoch 6/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4812 - loss: 0.7093 - val_accuracy: 0.3500 - val_loss: 0.7664
Epoch 7/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4812 - loss: 0.6597 - val_accuracy: 0.5000 - val_loss: 0.7743
Epoch 8/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4938 - loss: 0.6523 - val_accuracy: 0.

In [69]:
test_loss, test_accuracy = ckks_model.evaluate(test_dataset)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8000 - loss: 0.4194
Test loss: 0.41943296790122986
Test accuracy: 0.800000011920929


In [70]:
ckks_model.save("/content/ckks_friendly_model.keras")

In [72]:
dense1 = ckks_model.get_layer("dense_5")

weights1, bias1 = dense1.get_weights()

print("Weights shape:", weights1.shape)
print("Bias shape:", bias1.shape)

Weights shape: (7500, 32)
Bias shape: (32,)


In [73]:
for i, layer in enumerate(ckks_model.layers):
    print(i, layer.name, layer.__class__.__name__)

0 rescaling_2 Rescaling
1 flatten_2 Flatten
2 dense_5 Dense
3 lambda_2 Lambda
4 dense_6 Dense
5 lambda_3 Lambda
6 dense_7 Dense


In [74]:
dense1 = ckks_model.layers[2]

weights1, bias1 = dense1.get_weights()

print("Weights shape:", weights1.shape)
print("Bias shape:", bias1.shape)

Weights shape: (7500, 32)
Bias shape: (32,)


In [75]:
images, labels = next(iter(test_dataset))

image = images[0]

x = image.numpy() / 255.0
x = x.reshape(-1)

print("Input shape:", x.shape)

Input shape: (7500,)


In [76]:
plain_output = x @ weights1 + bias1

print("Plain output shape:", plain_output.shape)
print("First 5 outputs:", plain_output[:5])

Plain output shape: (32,)
First 5 outputs: [-0.96715075 -0.8481254   0.37151375 -1.4720873  -0.4343302 ]


In [77]:
encrypted_x = ts.ckks_vector(
    context,
    x.tolist()
)

print("Encrypted image created.")

Encrypted image created.


In [78]:
encrypted_output = []

for j in range(weights1.shape[1]):

    encrypted_neuron = encrypted_x.dot(
        weights1[:, j].tolist()
    )

    encrypted_neuron = encrypted_neuron + float(bias1[j])

    encrypted_output.append(encrypted_neuron)

print("Encrypted Dense calculation completed.")

Encrypted Dense calculation completed.


In [79]:
decrypted_output = np.array([
    encrypted_value.decrypt()[0]
    for encrypted_value in encrypted_output
])

print("Decrypted output shape:", decrypted_output.shape)
print("First 5 CKKS outputs:", decrypted_output[:5])

Decrypted output shape: (32,)
First 5 CKKS outputs: [-0.9671205  -0.84809638  0.37154503 -1.47205736 -0.43430112]


In [80]:
difference = np.abs(
    plain_output - decrypted_output
)

print("Plain output:")
print(plain_output[:5])

print("\nCKKS decrypted output:")
print(decrypted_output[:5])

print("\nMaximum difference:")
print(np.max(difference))

print("\nMean difference:")
print(np.mean(difference))

Plain output:
[-0.96715075 -0.8481254   0.37151375 -1.4720873  -0.4343302 ]

CKKS decrypted output:
[-0.9671205  -0.84809638  0.37154503 -1.47205736 -0.43430112]

Maximum difference:
3.401352176546357e-05

Mean difference:
3.0123480136587562e-05


In [83]:
encrypted_activation = []

for encrypted_value in encrypted_output:
    squared_value = encrypted_value * encrypted_value
    encrypted_activation.append(squared_value)

print("Encrypted polynomial activation completed.")

Encrypted polynomial activation completed.


In [84]:
decrypted_activation = np.array([
    value.decrypt()[0]
    for value in encrypted_activation
])

print("First 5 decrypted activation values:")
print(decrypted_activation[:5])

First 5 decrypted activation values:
[0.93532504 0.71926977 0.13804615 2.16695979 0.18861807]


In [85]:
plain_activation = plain_output ** 2

print("First 5 plaintext activation values:")
print(plain_activation[:5])

First 5 plaintext activation values:
[0.9353806  0.71931666 0.13802247 2.1670408  0.18864273]


In [86]:
activation_difference = np.abs(
    plain_activation - decrypted_activation
)

print("Maximum activation difference:")
print(np.max(activation_difference))

print("\nMean activation difference:")
print(np.mean(activation_difference))

Maximum activation difference:
0.00010851039642201243

Mean activation difference:
5.138577466987603e-05


In [87]:
dense2 = ckks_model.get_layer("dense_6")

weights2, bias2 = dense2.get_weights()

print("Weights shape:", weights2.shape)
print("Bias shape:", bias2.shape)

Weights shape: (32, 16)
Bias shape: (16,)


In [88]:
plain_dense2 = plain_activation @ weights2 + bias2

print("Plain second Dense shape:", plain_dense2.shape)
print("First 5 plaintext outputs:", plain_dense2[:5])

Plain second Dense shape: (16,)
First 5 plaintext outputs: [ 1.232743    0.08556227 -1.9026029   3.2187388   1.085421  ]


In [90]:
# Second Dense layer: encrypted activation -> 16 outputs
encrypted_dense2 = []

for j in range(weights2.shape[1]):

    # Start with the bias
    encrypted_neuron = ts.ckks_vector(
        context,
        [float(bias2[j])]
    )

    # Add encrypted x_i * weight_i
    for i in range(weights2.shape[0]):

        encrypted_neuron += encrypted_activation[i] * float(weights2[i, j])

    encrypted_dense2.append(encrypted_neuron)

print("Encrypted second Dense calculation completed.")

Encrypted second Dense calculation completed.


In [91]:
encrypted_activation = []

for encrypted_value in encrypted_output:
    squared_value = encrypted_value * encrypted_value

    # decrypt only to check? NO — don't do that for the real pipeline
    encrypted_activation.append(squared_value)

print(len(encrypted_activation))

32


In [92]:
encrypted_activation_vector = ts.ckks_vector(
    context,
    plain_activation.tolist()
)

print("Encrypted 32-value activation vector created.")

Encrypted 32-value activation vector created.


In [93]:
encrypted_dense2 = []

for j in range(weights2.shape[1]):

    encrypted_neuron = encrypted_activation_vector.dot(
        weights2[:, j].tolist()
    )

    encrypted_neuron = encrypted_neuron + float(bias2[j])

    encrypted_dense2.append(encrypted_neuron)

print("Second Dense calculation completed.")

Second Dense calculation completed.


In [94]:
encrypted_activation2 = []

for encrypted_value in encrypted_dense2:
    squared_value = encrypted_value * encrypted_value
    encrypted_activation2.append(squared_value)

print("Second encrypted polynomial activation completed.")

Second encrypted polynomial activation completed.


In [95]:
decrypted_activation2 = np.array([
    value.decrypt()[0]
    for value in encrypted_activation2
])

print("First 5 CKKS activation values:")
print(decrypted_activation2[:5])

First 5 CKKS activation values:
[1.51968064e+00 7.32208215e-03 3.61989437e+00 1.03603848e+01
 1.17816025e+00]


In [98]:
plain_activation2 = plain_dense2 ** 2

print("First 5 plaintext activation values:")
print(plain_activation2[:5])

First 5 plaintext activation values:
[1.5196553e+00 7.3209014e-03 3.6198978e+00 1.0360279e+01 1.1781387e+00]


In [99]:
activation2_difference = np.abs(
    plain_activation2 - decrypted_activation2
)

print("Plaintext:")
print(plain_activation2[:5])

print("\nCKKS decrypted:")
print(decrypted_activation2[:5])

print("\nMaximum difference:")
print(np.max(activation2_difference))

print("\nMean difference:")
print(np.mean(activation2_difference))

Plaintext:
[1.5196553e+00 7.3209014e-03 3.6198978e+00 1.0360279e+01 1.1781387e+00]

CKKS decrypted:
[1.51968064e+00 7.32208215e-03 3.61989437e+00 1.03603848e+01
 1.17816025e+00]

Maximum difference:
0.00010569489478839955

Mean difference:
3.0345596829013167e-05


In [100]:
dense3 = ckks_model.get_layer("dense_7")

weights3, bias3 = dense3.get_weights()

print("Weights shape:", weights3.shape)
print("Bias shape:", bias3.shape)

Weights shape: (16, 1)
Bias shape: (1,)


In [101]:
plain_final = plain_activation2 @ weights3 + bias3

print("Plain final output:")
print(plain_final)

Plain final output:
[-3.7702348]


In [104]:
print("Number of encrypted second activations:", len(encrypted_activation2))
print("Final weights shape:", weights3.shape)

Number of encrypted second activations: 16
Final weights shape: (16, 1)


In [107]:
print("weights3 shape:", weights3.shape)
print("bias3 shape:", bias3.shape)
print("plain_activation2 shape:", plain_activation2.shape)

weights3 shape: (16, 1)
bias3 shape: (1,)
plain_activation2 shape: (16,)


In [108]:
plain_final = np.dot(
    plain_activation2,
    weights3[:, 0]
) + bias3[0]

print("Plain final output:", plain_final)

Plain final output: -3.7702348


In [109]:
encrypted_final = (
    encrypted_activation2[0] * float(weights3[0, 0])
)

for i in range(1, 16):
    encrypted_final = (
        encrypted_final
        + encrypted_activation2[i] * float(weights3[i, 0])
    )

encrypted_final = encrypted_final + float(bias3[0])

print("Encrypted final Dense calculation completed.")

Encrypted final Dense calculation completed.


In [110]:
decrypted_final = encrypted_final.decrypt()[0]

print("Plain final output:", float(plain_final))
print("CKKS decrypted final output:", decrypted_final)

difference = abs(float(plain_final) - decrypted_final)

print("Absolute difference:", difference)

Plain final output: -3.7702348232269287
CKKS decrypted final output: -3.770347423450275
Absolute difference: 0.00011260022334624864


In [111]:
prediction = 1 if decrypted_final >= 0 else 0

print("Predicted class:", prediction)
print("True class:", true_label)

Predicted class: 0
True class: 0
